# Neural Network Quantization in Python: The Scale, Not the Bits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/neural_network_quantization.ipynb)

A float32 weight costs four bytes and an int8 weight costs one, so the usual question is how few bits you can get away with. On the network below, going from 8 bits to 3 costs 0.4 accuracy points. Staying at 2 bits and changing the rule that picks the scale moves accuracy from 0.129 to 0.925.

This notebook writes the quantization map in NumPy, applies it to a trained network, and measures which of the choices people argue about are worth anything.

Everything runs on a CPU in two to four minutes. The training is the slow part; the quantization is seconds.

Companion post: [Neural Network Quantization in Python: The Scale, Not the Bits](https://sesen.ai/blog/neural-network-quantization-python)

## 1. A network worth quantizing

MNIST and a small convolutional network. Eight epochs, which is where the budget curve flattens: quantizing an undertrained model turns every comparison below into a fact about training length rather than about quantization.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms

torch.set_num_threads(2)

tf = transforms.ToTensor()
train_ds = datasets.MNIST("data", train=True, download=True, transform=tf)
test_ds = datasets.MNIST("data", train=False, download=True, transform=tf)
x_train = train_ds.data.numpy()[:, None].astype(np.float32) / 255.0
y_train = train_ds.targets.numpy()
x_test = test_ds.data.numpy()[:, None].astype(np.float32) / 255.0
y_test = test_ds.targets.numpy()
print("train", x_train.shape, "test", x_test.shape)

In [ ]:
LAYERS = ("conv1", "conv2", "fc1", "fc2")


def make_model(seed=0):
    """Two convolutions and two fully connected layers.

    Deliberately lopsided: fc1 holds 51,264 of the 56,714 parameters. That is
    the same shape as a transformer, where the projections dwarf everything
    else, and it is what makes the per-layer sweep later say something.
    """
    torch.manual_seed(seed)
    m = nn.Sequential()
    m.add_module("conv1", nn.Conv2d(1, 16, 3))
    m.add_module("relu1", nn.ReLU())
    m.add_module("pool1", nn.MaxPool2d(2))
    m.add_module("conv2", nn.Conv2d(16, 32, 3))
    m.add_module("relu2", nn.ReLU())
    m.add_module("pool2", nn.MaxPool2d(2))
    m.add_module("flat", nn.Flatten())
    m.add_module("fc1", nn.Linear(32 * 5 * 5, 64))
    m.add_module("relu3", nn.ReLU())
    m.add_module("fc2", nn.Linear(64, 10))
    return m


def train(model, epochs=8, batch=128, seed=0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    xt, yt = torch.from_numpy(x_train), torch.from_numpy(y_train)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for idx in torch.randperm(len(xt), generator=g).split(batch):
            opt.zero_grad()
            lossf(model(xt[idx]), yt[idx]).backward()
            opt.step()
    model.eval()
    return model


def accuracy(model, batch=1000):
    model.eval()
    hits = 0
    with torch.no_grad():
        for i in range(0, len(x_test), batch):
            out = model(torch.from_numpy(x_test[i : i + batch]))
            hits += int((out.argmax(1).numpy() == y_test[i : i + batch]).sum())
    return hits / len(x_test)


t0 = time.time()
model = train(make_model(0))
baseline = accuracy(model)
print(f"float32 test accuracy {baseline:.4f}  ({time.time() - t0:.0f}s)")

## 2. The map

Two parameters and two lines:

```
q     = clip(round(w / s) + z, qmin, qmax)
w_hat = s * (q - z)
```

`s` is the scale, the width of one integer step in the units of the weight. `z` is the zero point, the integer that decodes to exactly 0.0. Symmetric quantization fixes `z = 0`; asymmetric fits it, which matters for one-sided values like ReLU outputs.

Biases are left in float32. There are 122 of them against 56,592 weights, so quantizing them saves nothing.

In [ ]:
def int_range(bits, symmetric=True):
    """Symmetric ranges drop the extra negative value so the grid is centred."""
    if symmetric:
        return -(2 ** (bits - 1)) + 1, 2 ** (bits - 1) - 1
    return 0, 2**bits - 1


def quantize_dequantize(w, s, bits, z=0.0):
    """q = clip(round(w/s) + z, qmin, qmax);  w_hat = s * (q - z)."""
    qmin, qmax = int_range(bits, symmetric=bool(np.all(z == 0.0)))
    q = np.clip(np.round(w / s) + z, qmin, qmax)
    return s * (q - z)


def weights(model):
    """The quantizable tensors as NumPy. Biases stay float32 throughout."""
    mods = dict(model.named_modules())
    return {n: mods[n].weight.detach().numpy().copy() for n in LAYERS}


def set_weights(model, tensors):
    mods = dict(model.named_modules())
    with torch.no_grad():
        for n, w in tensors.items():
            mods[n].weight.copy_(torch.from_numpy(np.ascontiguousarray(w)))
    return model


w = weights(model)
print({k: v.shape for k, v in w.items()})
print("weights:", sum(v.size for v in w.values()))

## 3. Three ways to pick the scale

`absmax` makes the largest magnitude representable and clips nothing, which sounds safe and is the one that breaks. `percentile` ignores the top 0.1% of magnitudes and lets them saturate. `mse` searches the clip that minimises the squared reconstruction error, starting from absmax, which is the ratio 1.0.

In [ ]:
def absmax_scale(w, bits, axis=None):
    """The largest magnitude lands on the last integer. Nothing clips."""
    _, qmax = int_range(bits)
    m = np.max(np.abs(w), axis=axis, keepdims=axis is not None)
    return np.maximum(m, 1e-12) / qmax


def percentile_scale(w, bits, pct=99.9, axis=None):
    """Ignore the top 0.1% of magnitudes. They saturate instead."""
    _, qmax = int_range(bits)
    a = np.abs(w)
    m = np.percentile(a, pct) if axis is None else np.percentile(a, pct, axis=axis, keepdims=True)
    return np.maximum(m, 1e-12) / qmax


def mse_scale(w, bits, axis=None, n_grid=80):
    """Search the clip that minimises squared error. Absmax is the point 1.0."""
    base = absmax_scale(w, bits, axis=axis)
    best_s, best_e = None, None
    for alpha in np.linspace(0.2, 1.0, n_grid):
        s = base * alpha
        e = (quantize_dequantize(w, s, bits) - w) ** 2
        e = e.sum() if axis is None else e.sum(axis=axis, keepdims=True)
        if best_e is None:
            best_e, best_s = e, np.broadcast_to(s, np.shape(e)).copy()
        else:
            take = e < best_e
            best_e, best_s = np.where(take, e, best_e), np.where(take, s, best_s)
    return best_s


RULES = {"absmax": absmax_scale, "percentile": percentile_scale, "mse": mse_scale}


def quantized_copy(model, bits, rule="absmax", per_channel=False, only=None):
    import copy

    q = copy.deepcopy(model)
    ws = weights(q)
    names = [only] if only else list(ws)
    out = {}
    for n in names:
        axis = tuple(range(1, ws[n].ndim)) if per_channel else None
        s = RULES[rule](ws[n], bits, axis=axis) if per_channel else RULES[rule](ws[n], bits)
        out[n] = quantize_dequantize(ws[n], s, bits)
    return set_weights(q, out)

## 4. The headline

Same network, same bit width, same rounding. The only difference is one number.

In [ ]:
for rule in ("absmax", "percentile", "mse"):
    print(f"2 bits, {rule:10s} {accuracy(quantized_copy(model, 2, rule)):.4f}")
print(f"float32              {baseline:.4f}")

## 5. The cliff

Read the curve twice. Between 8 bits and 3 the three rules are indistinguishable and the loss against float32 is a fraction of a point. At 2 bits they separate by 80 points.

In [ ]:
BITS = [8, 6, 4, 3, 2]
grid = {}
for bits in BITS:
    for rule in RULES:
        grid[(bits, rule)] = accuracy(quantized_copy(model, bits, rule))

fig, ax = plt.subplots(figsize=(7, 4))
for rule, colour in (("absmax", "#4a6d8c"), ("percentile", "#d99120"), ("mse", "#1f9e9e")):
    ax.plot(range(len(BITS)), [grid[(b, rule)] for b in BITS], "o-", color=colour, label=rule)
ax.axhline(baseline, color="grey", ls="--")
ax.set_xticks(range(len(BITS)))
ax.set_xticklabels(BITS)
ax.set_xlabel("bits per weight")
ax.set_ylabel("test accuracy")
ax.set_ylim(0, 1.05)
ax.legend(title="scale rule")
ax.grid(alpha=0.25)
plt.show()

## 6. Why, in one number

The accuracy is a consequence. The cause is how many weights land on the integer zero. At 2 bits there are three levels available, and if the step is set by the most distant weight, both non-zero levels sit outside the distribution entirely.

A network that is 99% zeros is not quantized, it is deleted.

In [ ]:
def zero_fraction(w, s, bits):
    qmin, qmax = int_range(bits)
    return float((np.clip(np.round(w / s), qmin, qmax) == 0).mean())


fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, rule in zip(axes, ("absmax", "mse")):
    s = float(RULES[rule](w["fc1"], 2))
    ax.hist(w["fc1"].ravel(), bins=160, color="#1f9e9e", alpha=0.6)
    for k in (-1, 0, 1):
        ax.axvline(k * s, color="#d99120" if k == 0 else "#4a6d8c", lw=1.6)
    ax.set_xlim(-0.45, 0.45)
    ax.set_yticks([])
    ax.set_title(f"2 bits, {rule}: step {s:.3f}, {zero_fraction(w['fc1'], s, 2):.0%} zeros")
plt.show()

for bits in BITS:
    row = []
    for rule in RULES:
        z = sum(
            zero_fraction(w[n], RULES[rule](w[n], bits), bits) * w[n].size for n in w
        ) / sum(v.size for v in w.values())
        row.append(f"{rule} {z:6.2%}")
    print(f"{bits} bits  " + "  ".join(row))

## 7. The tail that sets the scale

Absmax only fails when the largest weight is unrepresentative. Three of these four layers have no tail worth the name, with a largest weight within 8% of their 99.9th percentile. The fourth is the one holding 90% of the parameters.

This is the same shape LLM.int8() found in transformers past roughly 6.7B parameters, where a few feature dimensions carry outliers twenty times larger than the rest.

In [ ]:
for n, arr in w.items():
    a = np.abs(arr)
    kurt = ((arr - arr.mean()) ** 4).mean() / arr.var() ** 2
    print(
        f"{n:6s} n={arr.size:6d}  absmax={a.max():.4f}  p99.9={np.percentile(a, 99.9):.4f}"
        f"  ratio={a.max() / np.percentile(a, 99.9):.2f}  kurtosis={kurt:.2f}"
    )

## 8. Sparing a layer does not help

The usual next move is to find the fragile layer and keep it in float. Run both directions and watch the second one stay flat: at 2 bits every layer is independently sufficient to break the network, so protecting one rescues nothing. The fix is the scale, not the layer.

In [ ]:
print("one layer at 2 bits, the rest float32")
for n in LAYERS:
    print(f"  {n:6s} {accuracy(quantized_copy(model, 2, 'absmax', only=n)):.4f}")

print("one layer float32, the rest at 2 bits")
for spare in LAYERS:
    import copy

    q = copy.deepcopy(model)
    ws = weights(q)
    set_weights(
        q,
        {
            n: (arr if n == spare else quantize_dequantize(arr, absmax_scale(arr, 2), 2))
            for n, arr in ws.items()
        },
    )
    print(f"  {spare:6s} {accuracy(q):.4f}")

## 9. Activations, and how few images calibrate them

Weights are known before inference. Activation ranges are not, so they are estimated by pushing unlabelled data through and recording what comes out.

The usual question is how much data that needs. The answer here is eight images, because a range is a maximum over every element of every feature map, so one image already contributes tens of thousands of samples.

The second cell is where the zero point earns its keep: these are ReLU outputs, so a symmetric grid spends half its levels on values that cannot occur.

In [ ]:
def calibrate(model, x, pct=99.9):
    """Record each ReLU's output range over unlabelled data."""
    sites = {"relu1": [], "relu2": [], "relu3": []}
    mods = dict(model.named_modules())
    hooks = [
        mods[n].register_forward_hook(
            lambda _m, _i, o, k=n: sites[k].append(o.detach().numpy().ravel())
        )
        for n in sites
    ]
    with torch.no_grad():
        model(torch.from_numpy(x))
    for h in hooks:
        h.remove()
    return {
        k: float(np.max(np.concatenate(v)) if pct >= 100 else np.percentile(np.concatenate(v), pct))
        for k, v in sites.items()
    }


def with_act_quant(model, bits, ranges, asym=False):
    import copy

    q = copy.deepcopy(model)
    mods = dict(q.named_modules())
    for name, hi in ranges.items():
        qmin, qmax = int_range(bits, symmetric=not asym)
        s = max(hi / (qmax - qmin) if asym else hi / qmax, 1e-12)

        def hook(_m, _i, out, s=s, lo=qmin, up=qmax):
            return s * torch.clamp(torch.round(out / s), lo, up)

        mods[name].register_forward_hook(hook)
    return q


for n_calib in (8, 32, 128, 512, 2048):
    r = calibrate(model, x_train[:n_calib])
    print(f"{n_calib:5d} calibration images  int8 activations {accuracy(with_act_quant(model, 8, r)):.4f}")

print()
for label, pct, asym in (
    ("absmax, symmetric", 100.0, False),
    ("99.9th pct, symmetric", 99.9, False),
    ("absmax, asymmetric", 100.0, True),
    ("99.9th pct, asymmetric", 99.9, True),
):
    r = calibrate(model, x_train[:512], pct=pct)
    print(f"4-bit activations, {label:24s} {accuracy(with_act_quant(model, 4, r, asym)):.4f}")

## 10. What it buys

Bytes, immediately. Speed, only if someone has written the kernel. NumPy dispatches float32 to BLAS and has nothing equivalent for int8, so a naive integer matmul runs orders of magnitude slower. Production speedups come from inference stacks that fuse the dequantization into the multiply.

In [ ]:
n = sum(v.size for v in w.values())
for bits in (32, 8, 4, 2):
    print(f"{bits:2d}-bit weights  {n * bits / 8 / 1024:7.1f} KiB   x{32 / bits:.0f} smaller")
print(f"per-channel scales add {4 * sum(v.shape[0] for v in w.values())} bytes")

a = np.random.default_rng(0).normal(size=(1024, 800)).astype(np.float32)
b = np.random.default_rng(1).normal(size=(800, 64)).astype(np.float32)
for label, x, y in (("float32", a, b), ("int8", np.round(a * 8).astype(np.int8), np.round(b * 8).astype(np.int8))):
    t = time.perf_counter()
    for _ in range(20):
        x @ y
    print(f"{label:8s} matmul {(time.perf_counter() - t) / 20 * 1e6:9.1f} us")

## Exercises

1. **Find the ternary threshold.** At 2 bits the symmetric grid is exactly ternary. Li, Zhang and Liu (2016) derive an approximate optimum of `0.7 * mean(|w|)` assuming Gaussian weights. Compare that against what `mse_scale` finds for each layer, and check whether the gap tracks the kurtosis.
2. **Per-channel scales.** Give each output channel its own scale and re-run the sweep. It helps at 3 bits and destabilises 2. Work out why by counting how many weights each `conv1` channel has.
3. **Asymmetric weights.** Weights are roughly zero-centred, so a zero point should buy little. Confirm it, then re-run on a layer you have deliberately shifted by adding a constant.
4. **Break the calibration.** Calibrate the activation ranges on images of a single digit class rather than a random sample. How far does accuracy fall, and at which bit width does it start to matter?
5. **A real outlier.** Multiply one weight in `conv1` by 20 and retrain nothing. Measure what absmax does at 4 bits before and after, and then check whether a random rotation of the tensor undoes the damage.


## Further reading

- Jacob et al. (2018), [Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference](https://arxiv.org/abs/1712.05877)
- Krishnamoorthi (2018), [Quantizing deep convolutional networks for efficient inference](https://arxiv.org/abs/1806.08342), the whitepaper that surveys this design space
- Li, Zhang & Liu (2016), [Ternary Weight Networks](https://arxiv.org/abs/1605.04711)
- Dettmers et al. (2022), [LLM.int8()](https://arxiv.org/abs/2208.07339)
- Frantar et al. (2023), [GPTQ](https://arxiv.org/abs/2210.17323)
- [TurboQuant: How a Random Rotation Makes LLM Quantization Near-Optimal](https://sesen.ai/blog/turboquant-vector-quantization-random-rotations), the answer to the outlier problem this notebook measures
